# Imports

In [1]:
import cda2
#import datetime
import pyspark.sql.functions as F
import pyspark.sql.types as T
import json

from datetime import datetime, timedelta
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

# Connect to Spark

In [2]:
api = cda2.Api()

Set configuration parameters to better optimize queries.

In [3]:
config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "8g",
    "spark.executor.memoryOverhead": "16g",
}

Start Spark and specify number of cpus to use. 400 is quite high, but we'll be running 1 year at a time and want to have it done in just a few minutes.

In [4]:
#api.start_spark(n_executors=100, config=config)
api.start_spark(n_executors=100)

https://artifacts.mitre.org/artifactory/java-libs-release added as a remote repository with the name: repo-1
https://dali.mitre.org/nexus/content/repositories/mitre-caasd-releases added as a remote repository with the name: repo-2
https://dali.mitre.org/nexus/content/repositories/external-releases added as a remote repository with the name: repo-3
Ivy Default Cache set to: /home/rchong/.ivy2/cache
The jars for the packages stored in: /home/rchong/.ivy2/jars
org.mitre.spark#spark-geo_spark3.5_2.12 added as a dependency
org.apache.spark#spark-avro_2.12 added as a dependency
graphframes#graphframes added as a dependency
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
com.oracle.database.jdbc#ojdbc8 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9aad05bc-3890-44dc-bb6f-1bf0cdb9ceb8;1.0
	confs: [default]


:: loading settings :: url = jar:file:/devel/data_access/software/tdp-jupyter/poetry/cache/virtualenvs/python39-QwwvzYkJ-py3.9/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.mitre.spark#spark-geo_spark3.5_2.12;0.2.0 in repo-1
	found org.mitre.spark#spark-geo-core_2.12;0.2.0 in repo-1
	found org.scala-lang.modules#scala-collection-compat_2.12;2.11.0 in central
	found net.sf.geographiclib#GeographicLib-Java;2.0 in central
	found org.ejml#ejml-core;0.43.1 in central
	found org.ejml#ejml-ddense;0.43.1 in central
	found com.esri.geometry#esri-geometry-api;2.2.4 in central
	found com.fasterxml.jackson.core#jackson-core;2.9.6 in central
	found com.google.geometry#s2-geometry;2.0.0 in central
	found com.google.guava#guava;25.1-jre in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.checkerframework#checker-qual;2.0.0 in central
	found com.google.errorprone#error_prone_annotations;2.1.3 in central
	found com.google.j2objc#j2objc-annotations;1.1 in central
	found org.codehaus.mojo#animal-sniffer-annotations;1.14 in central
	found com.uber#h3;4.1.1 in central
	found org.apache.spark#spark-avro_2.12;3.5.1 in central
	found org.tuka

Function to convert Unix timestamp (milliseconds from 1970) to YYYYMMDD string.

In [5]:
# @F.udf("string")
# def to_date(ts):
#     return datetime.datetime.utcfromtimestamp(ts / 1000).strftime("%Y%m%d")

In [6]:
year0 = "2024"
year1 = str(int(year0) + 1)

In [7]:
dates = {"start_date": year0 + "-01-01", "end_date": year1 +"-01-01"}

In [8]:
print("retrieving fixes: ", datetime.now())

retrieving fixes:  2025-10-07 14:05:50.011253


In [9]:
# df_fixes_raw = (
#     api.dataframe("ArincFix", **dates, metadata=True)
#     .withColumn("uniq_fix_name", F.concat("identification.name", F.lit("("), "identification.icao_region", F.lit(")")))
#     .select(
#         "uniq_fix_name",
#         F.col("identification.name").alias("fix_name"),
#         F.col("arinc_record_info.customer_area_code").alias("area_code"),
#         F.col("identification.icao_region").alias("icao_region"),
#         "latitude",
#         "longitude",
#         "is_waypoint",
#         F.col("waypoint_info.type").alias("waypoint_type"),
#         F.col("waypoint_info.usage").alias("waypoint_usage"),
#         F.col("waypoint_info.name_format").alias("waypoint_name_format"),
#         F.col("waypoint_info.full_name").alias("waypoint_full_name"),
#         "is_vhf_navaid",
#         "is_ndb_navaid",
#         F.col("navaid_info.clazz").alias("navaid_class"),
#         F.col("navaid_info.facility_name").alias("navaid_facility_name"),
#         F.col("navaid_info.dme_latitude").alias("dme_latitude"),
#         F.col("navaid_info.dme_longitude").alias("dme_longitude"),
#         F.col("magnetic_variation.modeled").alias("magnetic_variation"),
#         F.col("metadata.effective_end_date").alias("end_date"),
#     )
#     .orderBy("uniq_fix_name")
# )

# #df_fixes_raw.show()

In [10]:
df_fixes_raw = (
    api.dataframe("ArincFix", **dates, metadata=True)
    .withColumn("uniq_fix_name", F.concat("identification.name", F.lit("("), "identification.icao_region", F.lit(")")))
    .select(
        "uniq_fix_name",
        F.col("identification.name").alias("fix_name"),
        F.col("arinc_record_info.customer_area_code").alias("area_code"),
        F.col("identification.icao_region").alias("icao_region"),
        "latitude",
        "longitude",
        F.col("navaid_info.dme_latitude").alias("dme_latitude"),
        F.col("navaid_info.dme_longitude").alias("dme_longitude"),
        F.col("magnetic_variation.modeled").alias("magnetic_variation"),
        F.col("metadata.effective_end_date").alias("end_date"),
    )
    .orderBy("uniq_fix_name")
)

#df_fixes_raw.show()

Multiple versions found: 3.1.64, 3.1.65, 3.1.67, 3.1.68, 3.1.69, 3.1.70, 3.1.71
                                                                                

In [11]:
#df_fixes_raw.count()

In [12]:
#df_fixes_raw.show()

In [13]:
print("   count starting: ", datetime.now())

   count starting:  2025-10-07 14:05:56.240752


In [14]:
window = Window.partitionBy("uniq_fix_name").orderBy(col("end_date").desc())

df_fixes = (df_fixes_raw
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row")
)
 
#df_fixes.show()

In [15]:
df_fixes.count()

279277

In [16]:
print("   count completed: ", datetime.now())

   count completed:  2025-10-07 14:06:20.492765


In [17]:
#df_fixes.show()

In [18]:
(
    df_fixes
    .repartition(1)
    .write.option("header", True)
    .csv("CRAFT/" + year0 + "/fixes", compression="None", mode="overwrite")
)

In [19]:
print("completed fixes: ", datetime.now())

completed fixes:  2025-10-07 14:06:29.352613
